In [ ]:
# SPR 2026 – BERTimbau MAX_LEN=512 + Raw Text (sem extração de seções)
# Estratégia: Usar texto bruto (clean_text apenas) em vez de extrair seções
# (Indicação/Achados/Comparativa). Hipótese: relatórios sem marcadores padrão
# perdem contexto no pré-processamento estruturado — raw text preserva tudo.
# Todo o resto idêntico ao winner (0.84027): model-v4, 5 folds, T=0.958, thresholds.

import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
MAX_LEN     = 512
NUM_CLASSES = 7
N_FOLDS     = 5
BATCH_SIZE  = 16
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v4/advanced_outputs_kaggle_4')
CONFIG_KEY   = 'bertimbau_large__cb_focal'
weights_dir  = WEIGHTS_BASE / 'weights' / CONFIG_KEY

# Thresholds e temperatura do winner (0.84027)
BEST_TEMP       = 0.9580
BEST_THRESHOLDS = [0.9500, 1.6000, 1.3500, 1.0, 0.4000, 1.1500, 0.1]

print(f'Device: {DEVICE}')
print('Modo: RAW TEXT (sem extração de seções)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item


def clean_text(text):
    """Limpeza mínima: remove caracteres de controle e normaliza espaços.
    NÃO extrai seções — preserva 100% do conteúdo do relatório.
    """
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRATION & INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        outputs = model(**kwargs)
        probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)


# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR DADOS DE TESTE — USA APENAS clean_text (sem build_input_text)
# ═══════════════════════════════════════════════════════════════════════════════
test_path  = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
test_df    = pd.read_csv(test_path)

# DIFERENÇA CHAVE: raw text sem extração de seções
test_texts = [clean_text(t) for t in test_df['report'].tolist()]
print(f'Test samples: {len(test_df)}')
print(f'Exemplo de texto (raw): {test_texts[0][:200]}...')

tokenizer = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')
dataset   = MammographyDataset(test_texts, tokenizer, MAX_LEN)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=0, pin_memory=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 5-FOLD ENSEMBLE INFERENCE (model-v4) — idêntico ao winner
# ═══════════════════════════════════════════════════════════════════════════════
config_path  = weights_dir / 'model_config'
test_probs   = np.zeros((len(test_df), NUM_CLASSES))
folds_loaded = 0

for fold in range(N_FOLDS):
    weight_path = weights_dir / f'fold_{fold}.pt'
    if not weight_path.exists():
        print(f'Fold {fold} não encontrado, pulando...')
        continue
    print(f'Carregando fold {fold}...', end=' ')
    config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_config(config).to(DEVICE)
    state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    fold_probs   = predict(model, loader)
    test_probs  += fold_probs
    folds_loaded += 1
    print(f'ok')
    del model, state_dict
    torch.cuda.empty_cache()

test_probs /= folds_loaded
print(f'{folds_loaded} folds carregados.')

# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO + SUBMISSÃO
# ═══════════════════════════════════════════════════════════════════════════════
calibrated_probs = temperature_scale(test_probs, BEST_TEMP)
predictions      = apply_thresholds(calibrated_probs, BEST_THRESHOLDS)

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'submission.csv salvo ({len(submission)} linhas)')